<a href="https://colab.research.google.com/github/MrPOLA/AES-LSB-Image-Steganography/blob/Pola/AES_Stegnography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Install required packages

In [1]:
!pip install Pillow pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.9 MB/s eta 0:00:00


# 📚 Import libraries

In [2]:
from PIL import Image
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad, unpad
import os
from google.colab import files

# 🔐 AES Encryption / Decryption

In [3]:
def encrypt(plaintext, key):
    cipher = AES.new(key, AES.MODE_EAX)
    nonce = cipher.nonce
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)
    return nonce, ciphertext, tag

def decrypt(nonce, ciphertext, tag, key):
    cipher = AES.new(key, AES.MODE_EAX, nonce=nonce)
    try:
        plaintext = cipher.decrypt_and_verify(ciphertext, tag)
        return plaintext
    except (ValueError, KeyError):
        print("Decryption failed: incorrect key or corrupted data")
        return None

# 🖼️ LSB Embedding

In [4]:
def embed_text(image_path, data):
    img = Image.open(image_path)
    if img.mode != 'RGBA':
        img = img.convert('RGBA')
    pixels = img.load()
    width, height = img.size

    binary_data = ''.join(f'{byte:08b}' for byte in data)
    data_len = len(binary_data)

    if data_len > width * height * 3:
        raise ValueError("Data too large to embed in the image.")

    data_index = 0
    for y in range(height):
        for x in range(width):
            r, g, b, a = pixels[x, y]
            if data_index < data_len:
                r = (r & ~1) | int(binary_data[data_index])
                data_index += 1
            if data_index < data_len:
                g = (g & ~1) | int(binary_data[data_index])
                data_index += 1
            if data_index < data_len:
                b = (b & ~1) | int(binary_data[data_index])
                data_index += 1
            pixels[x, y] = (r, g, b, a)
            if data_index >= data_len:
                break
        if data_index >= data_len:
            break
    return img

# 🔎 LSB Extraction

In [5]:
def extract_text(image_path, data_length):
    img = Image.open(image_path)
    if img.mode != 'RGBA':
        img = img.convert('RGBA')
    pixels = img.load()
    width, height = img.size

    binary_data = ""
    data_index = 0
    for y in range(height):
        for x in range(width):
            r, g, b, a = pixels[x, y]
            if data_index < data_length * 8:
                binary_data += str(r & 1)
                data_index += 1
            if data_index < data_length * 8:
                binary_data += str(g & 1)
                data_index += 1
            if data_index < data_length * 8:
                binary_data += str(b & 1)
                data_index += 1
            if data_index >= data_length * 8:
                break
        if data_index >= data_length * 8:
            break
    extracted_bytes = bytes(int(binary_data[i:i+8], 2) for i in range(0, len(binary_data), 8))
    return extracted_bytes

In [6]:
# Upload your image (this will appear in /content/)
uploaded = files.upload()

Saving Image.jpg to Image.jpg


In [7]:
# 📌 Define your inputs
image_path = "/content/Image.jpg"  # <- Image you uploaded
output_path = "/content/stego_image.png"
secret_message = b"This is a hidden message via AES + LSB"
key = get_random_bytes(16)  # Save this if you want to decrypt later

# 🧠 Encrypt the secret
nonce, ciphertext, tag = encrypt(secret_message, key)
data_to_embed = nonce + tag + ciphertext

# 🖼️ Embed in the image
stego_image = embed_text(image_path, data_to_embed)
stego_image.save(output_path)

print(f"✅ Message hidden successfully in: {output_path}")
print(f"🔑 Save this key to decrypt: {key.hex()}")

✅ Message hidden successfully in: /content/stego_image.png
🔑 Save this key to decrypt: 1bc4a5dc45f3882d211bd2aa49e73336


In [8]:
# 📦 Extract from image
embedded_length = len(nonce) + len(tag) + len(ciphertext)
extracted_data = extract_text(output_path, embedded_length)

# 🧩 Separate AES parts
ex_nonce = extracted_data[:16]
ex_tag = extracted_data[16:32]
ex_cipher = extracted_data[32:]

# 🔓 Decrypt
revealed = decrypt(ex_nonce, ex_cipher, ex_tag, key)

if revealed:
    print("✅ Decrypted message:", revealed.decode())
else:
    print("❌ Decryption failed.")

✅ Decrypted message: This is a hidden message via AES + LSB
